# Basic Cleaning — Shared preprocessing stage

## 1. Load

In [ ]:
import numpy as np
import pandas as pd

RAW_PATH = "../Dataset.csv"
OUT_PATH = "Preprocessed_Dataset.csv"

df = pd.read_csv(RAW_PATH)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded 6,178 rows x 74 columns


## 2. Drop identifiers

Names, contact details and record numbers identify individuals and carry no clinical signal.

In [ ]:
IDENTIFIERS = ["Person Id", "Full Name", "Contact Number", "Address Street",
               "Hospital Reference Number", "Ethnicity", "Gender"]

dropped_id = [c for c in IDENTIFIERS if c in df.columns]
df = df.drop(columns=dropped_id)
print("Dropped:", dropped_id)

Dropped: ['Person Id', 'Full Name', 'Contact Number', 'Address Street', 'Hospital Reference Number', 'Ethnicity', 'Gender']


## 3. Drop columns with a single value

In [ ]:
constant_cols = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]
df = df.drop(columns=constant_cols)
print("Dropped constant columns:", constant_cols)
print(f"Now {df.shape[1]} columns")

Dropped constant columns: ['Facilities for Daily Activity', 'First Aid Given At Seen']
Now 65 columns


## 4. Standardise text

Prevents the same answer typed with different spacing/capitalisation from counting as two
categories. Values stripped and lowercased; `"nan"`/`"none"` become real missing values, before
placeholder matching below.

In [ ]:
text_cols = [c for c in df.columns
             if df[c].dtype == object or str(df[c].dtype) == "str"]

for c in text_cols:
    df[c] = df[c].astype(str).str.strip().str.lower()
    df.loc[df[c].isin(["nan", "none", ""]), c] = np.nan

print(f"Standardised {len(text_cols)} text columns")

Standardised 65 text columns


### 4a. Merge spelling variants

A few free-text answers differ only by spacing — merged to keep category counts honest.

In [ ]:
SPELLING_FIXES = {
    "for xray": "for x-ray",
    "for neurosurgical opinion": "for neuro surgical opinion",
}

fixed = 0
for c in text_cols:
    if c not in df.columns:
        continue
    before = df[c].isin(SPELLING_FIXES).sum()
    if before:
        df[c] = df[c].replace(SPELLING_FIXES)
        fixed += before
print(f"Merged {fixed} values into their canonical spelling")

Merged 2 values into their canonical spelling


## 5. Placeholder phrases

Data collectors typed explanatory sentences instead of leaving cells blank. These are missing
values carrying a reason, not categories.

**One phrase was previously unmapped:** `"early discharge/ data entry staff missed to enter"`,
679 times (11%) in `Management Name 1` — the second most common "treatment" if left as text.
Included below.

In [ ]:
PLACEHOLDER_MAP = {
    "victim not willing to share/ unable to respond/  early discharge": "patient_unable_unwilling",
    "victim doesn't have knowledge on distance/ not willing to share/ unable to respond/  early discharge": "patient_has_no_knowledge",
    "victim unable to recall the time or early discharge": "patient_unable_unwilling",
    "victim not aware about the time/ not willing to share/ unable to respond/  early discharge": "patient_unable_unwilling",
    "early discharge/ unable to look the bht": "early_discharge",
    "early discharge/ unable to access the bht": "early_discharge",
    "early discharge/ data entry staff missed to enter": "early_discharge",
    "data missing": "data_entry_gap",
    "missing data": "data_entry_gap",
    "not necessary": "not_applicable",
    "not necessary for student and unemployed": "not_applicable",
    "na": "not_applicable",
    "unknown": "data_entry_gap",
}
print(f"{len(PLACEHOLDER_MAP)} placeholder phrases mapped to "
      f"{len(set(PLACEHOLDER_MAP.values()))} reasons")

13 placeholder phrases mapped to 5 reasons


### 5a. Check no placeholder-like text was missed

Every remaining category is scanned for wording resembling an explanation of absence, rather than
trusting the map to be complete.

In [ ]:
HINTS = ["victim", "early discharge", "not willing", "unable", "not aware", "no knowledge",
         "not necessary", "missing", "unknown", "missed", "data entry", "n/a", "not applicable"]

suspects = []
for c in text_cols:
    if c not in df.columns:
        continue
    for value, n in df[c].value_counts().items():
        text = str(value)
        if text in PLACEHOLDER_MAP:
            continue
        if any(h in text for h in HINTS):
            suspects.append({"column": c, "value": text[:60], "rows": n})

if suspects:
    print("REVIEW - possible unmapped placeholders:")
    print(pd.DataFrame(suspects).to_string(index=False))
else:
    print("No unmapped placeholder-like text found.")

No unmapped placeholder-like text found.


## 6. Convert placeholders to missing, recording the reason

`list(df.columns)` is snapshotted before the loop, so reason columns created inside it don't get
their own reason columns.

In [ ]:
original_cols = list(df.columns)

for col in original_cols:
    reason_col = f"{col}_missing_reason"
    df[reason_col] = "not_missing"
    for phrase, reason in PLACEHOLDER_MAP.items():
        mask = df[col] == phrase
        df.loc[mask, reason_col] = reason
        df.loc[mask, col] = np.nan
    blank = df[col].isna() & (df[reason_col] == "not_missing")
    df.loc[blank, reason_col] = "data_entry_gap"

print(f"{len(original_cols)} data columns + {len(original_cols)} reason columns")

reasons = pd.concat([df[f"{c}_missing_reason"] for c in original_cols])
print("\nReason distribution across all cells:")
print(reasons.value_counts().to_string())

65 data columns + 65 reason columns

Reason distribution across all cells:
not_missing                 242613
patient_unable_unwilling     77778
not_applicable               41743
early_discharge              20502
data_entry_gap               15320
patient_has_no_knowledge      3614


## 7. Tidy the age bands

Drops the trailing `y` (`"1-9y"` → `"1-9"`) so bands sort and read cleanly. Missing stays
missing.

In [ ]:
df["Age"] = df["Age"].str.replace("y", "", regex=False)
print(df["Age"].value_counts(dropna=False).to_string())

Age
20-29     1475
30-39     1107
NaN       1000
40-49      851
50-59      512
10-19      393
60-69      376
70-79      207
1-9        202
80-89       50
90-100       5


## 8. Validate before saving

Assertions to catch a silent breakage if this notebook is re-run after the raw file changes. Six
notebooks depend on this output.

In [ ]:
data_cols = [c for c in df.columns if not c.endswith("_missing_reason")]
reason_cols = [c for c in df.columns if c.endswith("_missing_reason")]

assert len(data_cols) == len(reason_cols), "every data column needs one reason column"
assert not any(c.count("_missing_reason") > 1 for c in df.columns), "reason-of-reason column"
assert df.shape[0] == 6178, f"row count changed: {df.shape[0]}"
assert not any(c in df.columns for c in IDENTIFIERS), "identifier survived"

for c in data_cols:
    present = df[f"{c}_missing_reason"] == "not_missing"
    assert df.loc[present, c].notna().all(), f"{c}: marked not_missing but value is blank"
    assert df.loc[~present, c].isna().all(), f"{c}: marked missing but has a value"

print(f"All checks passed - {len(data_cols)} data + {len(reason_cols)} reason columns")

All checks passed - 65 data + 65 reason columns


## 9. Save

In [ ]:
df.to_csv(OUT_PATH, index=False)
print(f"Saved {OUT_PATH}  {df.shape}")

Saved Preprocessed_Dataset.csv  (6178, 130)
